In [ ]:
!pip install colab-ssh --upgrade
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared("your_password")

In [ ]:

!pip install fastapi uvicorn pyngrok nest-asyncio transformers torch

: 

In [ ]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.nn.functional as F
from pyngrok import ngrok
import uvicorn
import os

# 2. 구글 드라이브 마운트 및 모델 로드
from google.colab import drive
drive.mount('/content/drive')

# 사용자 지정 경로 설정
MODEL_PATH = "/content/drive/MyDrive/models/ga1_model"

if not os.path.exists(MODEL_PATH):
    print(f"⚠️ 모델 경로를 확인해주세요: {MODEL_PATH}")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 모델 및 토크나이저 로드 (Bert 전용 클래스 사용)
    model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
    tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
    model.to(device)
    model.eval()
    print(f"✅ GA1 모델 로드 완료 (Device: {device})")

# 3. FastAPI 앱 정의
app = FastAPI()

class TextRequest(BaseModel):
    text: str 

@app.post("/predict")
async def predict_endpoint(data: TextRequest):
    inputs = tokenizer(
        data.text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)
        prediction = torch.argmax(logits, dim=-1).item()
        confidence = probs[0][prediction].item()
    
    return {
        "prediction": prediction, 
        "confidence": confidence
    }

# 4. ngrok 설정 및 실행
# ⚠️ 여기에 본인의 실제 ngrok 토큰을 입력하세요
NGROK_AUTH_TOKEN = "39H7VwX2xfGhL2Fz01D4IVJ85lb_49FjgFGznHHyuvpzXxJ3T" 
ngrok.kill() # 기존에 실행 중인 ngrok 프로세스가 있다면 종료
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 포트 8000번으로 터널링 생성
public_url = ngrok.connect(8000)
print(f"\n🚀 Colab 서버가 실행되었습니다!")
print(f"🔗 아래 주소를 복사하여 로컬 'ga1_safety.py'의 remote_url에 넣으세요:")
print(f"{public_url.public_url}/predict\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')